# Part 1 · Basic RAG on one document

You build retrieval by hand, in memory, on one Markdown document, so every step is visible: chunks, embeddings, dense
search, BM25, rank fusion. Then you wrap the search in a tool and give it to a Deep Agent that answers with citations.
Notebook 02 moves the same ideas to a real vector database (Qdrant) and real Vietnamese PDFs.

The document `data/ev_battery_guide.md` (*Hướng dẫn sử dụng và bảo dưỡng pin xe điện*, 10 sections) is a **sample
document written for this lab**: general guidance, illustrative error codes, no brand claims.

| Step | What you see |
|---|---|
| 1. Load and chunk | sections from `##` headings, chunks of about 600 characters with metadata |
| 2. Embed | each chunk becomes a 1536-number vector |
| 3. Dense search | cosine similarity: finds paraphrases |
| 4. Sparse search (BM25) | word overlap: finds exact codes and names |
| 5. Hybrid (RRF) | both rankings fused by rank |
| 6. Tool | `search_guide(query)` returns numbered passages |
| 7. Agent | a Deep Agent searches, then answers with citations, or says the guide does not cover it |

In [8]:
import logging
import os
import re
import time

import numpy as np
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

load_dotenv()  # the .env of this session folder
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing: copy .env.example to .env and fill it in (README, setup step 3).")
MODEL = os.getenv("MODEL", "openai:gpt-6-luna")
logger.info("Model: %s", MODEL)

from utils import SESSION_DIR, log_call, short, stream, system_prompt, table  # helpers shared by both notebooks

INFO | Model: openai:gpt-6-luna


## 1. Load the document and split it into chunks

A model can only read a few passages per question, so the document is cut into **chunks** that are searched one by one.
Two splitters, the usual LangChain recipe for Markdown:
- [`MarkdownHeaderTextSplitter`](https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter)
  cuts at the headings and keeps them as metadata (`section`), so every chunk knows where it comes from (used later for citations);
- [`RecursiveCharacterTextSplitter`](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter)
  cuts long sections into pieces of about 600 characters, preferring paragraph and line breaks, with up to 150
  characters of overlap so a sentence cut at a boundary still appears whole in one of the two chunks.

How the overlap is built: the splitter first cuts the text at the separators (`"\n\n"`, then `"\n"`, then `" "`), then
glues those pieces back together up to `chunk_size`. When a chunk is full, the next chunk starts with the **last whole
pieces** of the previous one, as long as they fit in `chunk_overlap`. It never cuts a piece to fill the overlap. The
lines of this guide are about 120 characters long, so `chunk_overlap=100` repeats nothing (no line fits in 100
characters), while 150 repeats one line. The cell prints the text shared by two consecutive chunks to check it.

<img src="images/b5_ingest_flow.png" width="640">

The ingest flow of this notebook (load → parse → chunk → embed → store).

In [10]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

GUIDE = SESSION_DIR / "data" / "ev_battery_guide.md"
text = GUIDE.read_text(encoding="utf-8")

sections = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "title"), ("##", "section")]).split_text(text)
# overlap = whole trailing pieces (here: lines of ~120 characters) that fit in chunk_overlap; 100 would repeat nothing
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=150)
chunks = splitter.split_documents(sections)
for chunk in chunks:
    chunk.metadata["source"] = GUIDE.name
    chunk.metadata.setdefault("section", chunk.metadata["title"])  # the note before the first ## heading

print(f"{len(text):,} characters → {len(sections)} sections → {len(chunks)} chunks")
for chunk in chunks[1:3]:
    print(f"\n{chunk.metadata}\n{short(chunk.page_content, 10000)}")


def shared_text(a: str, b: str) -> str:
    """Longest end of chunk a that is also the start of chunk b: the overlap."""
    return next((b[:k] for k in range(min(len(a), len(b)), 0, -1) if a.endswith(b[:k])), "")


pairs = [(a, b) for a, b in zip(chunks, chunks[1:]) if a.metadata["section"] == b.metadata["section"]]
sizes = [len(shared_text(a.page_content, b.page_content)) for a, b in pairs]
print(f"\noverlap between consecutive chunks of the same section (characters): {sizes}")
a, b = pairs[0]
print(f"repeated text: {shared_text(a.page_content, b.page_content)!r}")

11,455 characters → 11 sections → 26 chunks

{'title': 'Hướng dẫn sử dụng và bảo dưỡng pin xe điện', 'section': '1. Giới thiệu và phạm vi', 'source': 'ev_battery_guide.md'}
Pin là bộ phận đắt nhất của một chiếc xe điện và quyết định quãng đường đi được, tốc độ sạc cũng như giá trị bán lại của xe. Một bộ pin được dùng đúng cách có thể giữ phần lớn dung lượng sau nhiều năm; một bộ pin thường xuyên bị sạc đầy rồi để nóng dưới nắng sẽ xuống cấp nhanh hơn đáng kể. Hướng dẫn này áp dụng cho ô tô điện và xe máy điện dùng pin lithium-ion. Tài liệu giải thích sự khác nhau giữa các loại pin, các cách sạc, mức sạc nên dùng hằng ngày, ảnh hưởng của nhiệt độ, cách cất giữ xe khi không dùng, chính sách bảo

{'title': 'Hướng dẫn sử dụng và bảo dưỡng pin xe điện', 'section': '1. Giới thiệu và phạm vi', 'source': 'ev_battery_guide.md'}
pin, các cách sạc, mức sạc nên dùng hằng ngày, ảnh hưởng của nhiệt độ, cách cất giữ xe khi không dùng, chính sách bảo hành thường gặp, ý nghĩa của các mã lỗi và những qu

## 2. Embed the chunks

An **embedding model** turns a text into a vector; texts with similar meaning get vectors pointing in similar
directions, even with different words. [`OpenAIEmbeddings`](https://docs.langchain.com/oss/python/integrations/text_embedding/openai)
with `text-embedding-3-small` returns 1536 numbers per text ([OpenAI guide](https://platform.openai.com/docs/guides/embeddings)).
`embed_documents` embeds the chunks in one batch call; `embed_query` embeds a question. The vectors are already
normalised to length 1, so a dot product is the cosine similarity.

In [3]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
started = time.monotonic()
chunk_vectors = np.array(embeddings.embed_documents([c.page_content for c in chunks]))
logger.info("Embedded %d chunks in %.1f s", len(chunks), time.monotonic() - started)

print("matrix shape:", chunk_vectors.shape)  # (chunks, dimensions)
print("first chunk, first 5 values:", np.round(chunk_vectors[0][:5], 4))
print("vector length:", round(float(np.linalg.norm(chunk_vectors[0])), 3))

INFO | Embedded 26 chunks in 0.7 s


matrix shape: (26, 1536)
first chunk, first 5 values: [ 0.022   0.0579 -0.0444 -0.0528  0.0613]
vector length: 1.0


## 3. Dense search: cosine similarity

Embed the question, compare it with every chunk vector, keep the top `k`. This is what a vector database does, at scale
and with an index instead of a full scan. Dense search shines on **paraphrases**: the question below never says
*bảo quản* or *không sử dụng lâu ngày*, the words of section 7.

In [4]:
def dense_search(query: str, k: int = 4) -> list[tuple[int, float]]:
    """(chunk index, cosine similarity) of the k chunks closest to the query."""
    scores = chunk_vectors @ np.array(embeddings.embed_query(query))
    best = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in best]


def show(hits: list[tuple[int, float]]) -> None:
    table([[rank, f"{score:.3f}", chunks[i].metadata["section"][:38], short(chunks[i].page_content, 60)]
           for rank, (i, score) in enumerate(hits, 1)], ["rank", "score", "section", "preview"])


paraphrase = "Để xe lâu không chạy thì nên làm gì với pin?"
show(dense_search(paraphrase))

rank  score  section                                 preview                                                      
1     0.719  10. An toàn khi sạc và khi xảy ra sự c  xe điện. - Nếu thấy khói, mùi khét hoặc tiếng nổ lách tách t…
2     0.717  7. Bảo quản khi không sử dụng lâu ngày  Khi xe không được dùng trong thời gian dài (đi công tác, xe …
3     0.695  7. Bảo quản khi không sử dụng lâu ngày  sạc bổ sung về khoảng 50% nếu cần. - Nếu xe có chế độ cất gi…
4     0.684  10. An toàn khi sạc và khi xảy ra sự c  Pin xe điện chứa năng lượng lớn ở điện áp cao (thường 400 V …


## 4. Sparse search: BM25

[BM25](https://en.wikipedia.org/wiki/Okapi_BM25) scores a chunk by the query words it contains: rare words count more
(IDF), repeated words count with diminishing returns, long chunks are normalised. Each text becomes a *sparse* vector:
one weight per word of the vocabulary, almost all zero. It needs no model, only a **tokenizer**. The regex `\w+`
keeps Vietnamese letters with their diacritics (`lỗi`, `điện`) and splits `E-305` into `e` and `305`.
We use [`rank_bm25.BM25Okapi`](https://github.com/dorianbrown/rank_bm25).

Two questions show why both retrievers are useful:
- **keyword** `mã lỗi E-305`: the answer is in the chunk that contains the code; BM25 finds it, dense search ranks
  another chunk of the error-code section first (every error code "means" roughly the same thing to an embedding);
- **paraphrase** (step 3): BM25 matches common words (`xe`, `pin`) and puts section 7 lower.

In [5]:
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())  # \w is Unicode-aware: keeps ă, ơ, ư, đ and the tone marks


bm25 = BM25Okapi([tokenize(c.page_content) for c in chunks])


def bm25_search(query: str, k: int = 4) -> list[tuple[int, float]]:
    scores = bm25.get_scores(tokenize(query))
    best = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in best]


keyword = "mã lỗi E-305"
print("tokens:", tokenize(keyword))
for name, search in [("dense", dense_search), ("bm25", bm25_search)]:
    hits = search(keyword)
    top = chunks[hits[0][0]].page_content
    print(f"\n{name}: top chunk contains 'E-305'? {'E-305' in top}")
    show(hits)

print(f"\nbm25 for the paraphrase: {paraphrase}")
show(bm25_search(paraphrase))

tokens: ['mã', 'lỗi', 'e', '305']

dense: top chunk contains 'E-305'? False
rank  score  section                                 preview                                                      
1     0.532  9. Mã lỗi pin thường gặp (minh họa)     Khi BMS phát hiện bất thường, màn hình hiển thị cảnh báo kèm…
2     0.530  9. Mã lỗi pin thường gặp (minh họa)     - **E-412: Phát hiện rò điện cách điện.** Đây là lỗi an toàn…
3     0.506  9. Mã lỗi pin thường gặp (minh họa)     - **E-217: Nhiệt độ pin quá cao khi sạc nhanh.** Xe tự giảm …
4     0.398  Hướng dẫn sử dụng và bảo dưỡng pin xe   > Tài liệu mẫu viết cho bài thực hành (Session 05). Nội dung…

bm25: top chunk contains 'E-305'? True
rank  score  section                                 preview                                                      
1     7.407  9. Mã lỗi pin thường gặp (minh họa)     - **E-217: Nhiệt độ pin quá cao khi sạc nhanh.** Xe tự giảm …
2     7.352  9. Mã lỗi pin thường gặp (minh họa)     - **E-412: Phát hiện rò đi

## 5. Hybrid search: Reciprocal Rank Fusion

Dense scores (cosine, 0 to 1) and BM25 scores (unbounded) are on different scales, so they are not added. **Reciprocal
Rank Fusion** ([Cormack et al., 2009](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)) uses only the ranks:
each list gives a chunk `1 / (k + rank)`, and the sums are sorted. `k = 60` (the paper's value) damps the difference
between rank 1 and rank 2, so a chunk that is good in both lists beats one that is first in a single list.
Qdrant's `FusionQuery(RRF)` in notebook 02 does the same on the server. A chunk ranked 1st by one retriever and 2nd
by the other gets the same score as the chunk with the opposite ranks: ties are normal with RRF.

The last table answers "where does the right chunk land?" for the two questions. Each retriever puts the right chunk
lower on one of them; hybrid keeps it near the top on both. RRF is a robust default, not always the best single ranking.

<img src="images/b5_rrf_example.png" width="640">

RRF on a toy example: chunks ranked high in both lists rise to the top.

In [6]:
def rrf(rankings: list[list[tuple[int, float]]], k: int = 60) -> list[tuple[int, float]]:
    """Fuse ranked lists of (chunk index, score) by rank: sum of 1 / (k + rank)."""
    fused: dict[int, float] = {}
    for ranking in rankings:
        for rank, (i, _) in enumerate(ranking, 1):
            fused[i] = fused.get(i, 0.0) + 1 / (k + rank)
    return sorted(fused.items(), key=lambda item: -item[1])


def hybrid_search(query: str, k: int = 4) -> list[tuple[int, float]]:
    return rrf([dense_search(query, k=10), bm25_search(query, k=10)])[:k]


for query in (keyword, paraphrase):
    print(f"\nhybrid: {query}")
    show(hybrid_search(query))

# Where does the right chunk land in each ranking? (the chunk with the code / a chunk of section 7)
relevant = {keyword: lambda c: "E-305" in c.page_content, paraphrase: lambda c: c.metadata["section"].startswith("7.")}


def first_relevant(hits, is_relevant) -> int | str:
    return next((rank for rank, (i, _) in enumerate(hits, 1) if is_relevant(chunks[i])), "-")


print("\nrank of the first relevant chunk (top 4)")
table([[q, *(first_relevant(search(q), ok) for search in (dense_search, bm25_search, hybrid_search))]
       for q, ok in relevant.items()], ["question", "dense", "bm25", "hybrid"])


hybrid: mã lỗi E-305
rank  score  section                                 preview                                                      
1     0.032  9. Mã lỗi pin thường gặp (minh họa)     Khi BMS phát hiện bất thường, màn hình hiển thị cảnh báo kèm…
2     0.032  9. Mã lỗi pin thường gặp (minh họa)     - **E-217: Nhiệt độ pin quá cao khi sạc nhanh.** Xe tự giảm …
3     0.032  9. Mã lỗi pin thường gặp (minh họa)     - **E-412: Phát hiện rò điện cách điện.** Đây là lỗi an toàn…
4     0.031  Hướng dẫn sử dụng và bảo dưỡng pin xe   > Tài liệu mẫu viết cho bài thực hành (Session 05). Nội dung…

hybrid: Để xe lâu không chạy thì nên làm gì với pin?
rank  score  section                                 preview                                                      
1     0.032  6. Ảnh hưởng của nhiệt độ               tiếp nhiều lần khi trời nóng, và để hệ thống làm mát pin hoạ…
2     0.032  7. Bảo quản khi không sử dụng lâu ngày  sạc bổ sung về khoảng 50% nếu cần. - Nếu xe có chế độ cất gi…
3   

## 6. A search tool for the agent

The agent does not call `hybrid_search` directly: it gets a **tool** ([LangChain tools](https://docs.langchain.com/oss/python/langchain/tools)).
The docstring is the description the model reads to decide when to call it and what to pass. The tool returns the
top 4 passages numbered `[n] section — text`, so the answer can point to them. One log line per call (`INFO | search_guide(...) -> ok`).

<img src="images/nb_query_flow.png" width="640">

The query flow: the model writes the query in a tool call, the tool returns numbered passages to cite.

In [7]:
from langchain.tools import tool


@tool(parse_docstring=True)
def search_guide(query: str) -> str:
    """Search the EV battery guide (Vietnamese) with hybrid search (meaning + keywords).

    Returns the 4 best passages as "[n] section — text".

    Args:
        query: What to look for, in Vietnamese, e.g. "mức sạc khuyến nghị pin NMC".
    """
    started, status = time.monotonic(), "error"
    try:
        hits = hybrid_search(query, k=4)
        status = "ok"
        return "\n\n".join(f"[{n}] {chunks[i].metadata['section']} — {chunks[i].page_content}"
                            for n, (i, _) in enumerate(hits, 1))
    finally:
        log_call("search_guide", repr(query), status, started)


print(search_guide.invoke({"query": "E-217"})[:400])

INFO | search_guide('E-217') -> ok 191 ms


[1] 9. Mã lỗi pin thường gặp (minh họa) — - **E-217: Nhiệt độ pin quá cao khi sạc nhanh.** Xe tự giảm hoặc dừng công suất sạc DC. Dừng sạc, để xe nghỉ ở nơi
thoáng mát 15 đến 30 phút rồi sạc lại với công suất thấp hơn. Nếu lỗi xuất hiện khi trời không nóng, cần kiểm tra hệ
thống làm mát pin.
- **E-305: Chênh lệch điện áp giữa các cụm cell vượt ngưỡng.** BMS phát hiện các cụm cell không cân bằng. S


## 7. A Deep Agent that answers from the guide

[`create_deep_agent`](https://docs.langchain.com/oss/python/deepagents/overview) builds the agent loop (model → tools →
model) with the planning and file tools of Deep Agents; we add `search_guide`. The behaviour (answer only from search
results, cite, say when the documents do not cover a question) lives in the shared prompt
[`prompts/system_prompt.md`](../prompts/system_prompt.md), used by both notebooks. See also the
[RAG with Deep Agents](https://docs.langchain.com/oss/python/deepagents/rag) guide.

`stream()` (in `utils.py`) streams the run ([streaming](https://docs.langchain.com/oss/python/langchain/streaming)):
`→` a tool call, `←` its result, then the answer token by token. `INFO |` lines are logs.

In [8]:
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

agent = create_deep_agent(model=init_chat_model(MODEL), tools=[search_guide], system_prompt=system_prompt())

A question in everyday words (no term from the guide), then a question the guide does not cover. Change `question` and
run the cell again to ask your own.

In [9]:
question = "Tôi sắp đi công tác một tháng, nên để xe điện thế nào cho pin bền?"
answer = stream(agent, question)

Q: Tôi sắp đi công tác một tháng, nên để xe điện thế nào cho pin bền?
→ search_guide({"query": "Để xe điện không sử dụng một tháng, lưu trữ pin thế nào để pin bền? mức sạc, nhiệt độ, kiểm tra địn…)


INFO | search_guide('Để xe điện không sử dụng một tháng, lưu trữ pin thế nào để pin bền? mức sạc, nh…) -> ok 253 ms


← search_guide success: [1] 7. Bảo quản khi không sử dụng lâu ngày — Khi xe không được dùng trong thời gian dài (đi công tác, xe dự ph…
Trước khi cất xe, hãy để pin ở mức **40–60%**; không để pin đầy 100% hoặc cạn. Đỗ xe ở nơi khô ráo, thoáng mát, tránh nắng trực tiếp; nếu xe có chế độ cất giữ hoặc vận chuyển, hãy bật chế độ đó. [1]  
Vì xe vẫn tự tiêu thụ một ít điện, hãy kiểm tra mức pin mỗi tháng một lần và sạc bổ sung về khoảng 50% nếu cần. [1]  
Nếu là xe máy điện có pin tháo rời, hãy cất pin trong nhà ở nhiệt độ phòng, tránh xa nguồn nhiệt. [3]


In [10]:
question = "Bao lâu thì nên thay lốp xe điện?"  # not in the guide
answer = stream(agent, question)

Q: Bao lâu thì nên thay lốp xe điện?
→ search_guide({"query": "bao lâu nên thay lốp xe điện thời hạn thay lốp tuổi thọ lốp xe điện"})


INFO | search_guide('bao lâu nên thay lốp xe điện thời hạn thay lốp tuổi thọ lốp xe điện') -> ok 289 ms


← search_guide success: [1] 8. Suy giảm dung lượng và bảo hành — hành của xe. Một số điểm cần lưu ý: - Bảo hành thường không áp dụng k…
→ search_guide({"query": "lốp xe điện độ mòn rãnh gai thay lốp tuổi lốp kilomet kiểm tra lốp xe electric"})


INFO | search_guide('lốp xe điện độ mòn rãnh gai thay lốp tuổi lốp kilomet kiểm tra lốp xe electric') -> ok 211 ms


← search_guide success: [1] 9. Mã lỗi pin thường gặp (minh họa) — - **E-217: Nhiệt độ pin quá cao khi sạc nhanh.** Xe tự giảm hoặc dừn…
Tài liệu hiện có không nêu thời hạn hay số ki-lô-mét cố định để thay lốp xe điện. Vì vậy, chưa đủ thông tin để xác định bao lâu thì nên thay; cần hướng dẫn của nhà sản xuất lốp/xe hoặc thông tin về tình trạng lốp hiện tại.


**What to remember**
- Chunking decides what the model can see: sections as metadata give every chunk a citation for free.
- Dense search finds meaning, BM25 finds exact words (codes, names, numbers); RRF combines them by rank.
- Retrieval is a tool: the agent decides when to search and what to search for, and can search again.

Next: notebook 02 does the same with a Qdrant server, scanned PDFs, tables, access control and document versions.